In [2]:
# -*- coding: utf-8 -*-
"""
Cross-dataset transfer between WDP, WAD20 and R&G
=================================================

Trains CLIP-Base on one full dataset (with a 15% validation split held
out for early stopping) and evaluates it, without any fine-tuning, on the
other two datasets. Only the off-diagonal cells of the transfer matrix
are computed here; the within-dataset numbers come from the separate
5-fold CV notebooks. Ratings are z-scored per dataset, so the reported
Pearson r measures agreement in ranking rather than absolute calibration.

Transfer pairs: WDP -> WAD20, R&G; WAD20 -> WDP, R&G; R&G -> WDP, WAD20.

Source of the transfer matrix in Table 6 of the paper.

Written for Google Colab. Expects the three dataset archives under
/content/drive/MyDrive/datasets/.
"""

import os
import random
import json
from dataclasses import dataclass
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torch import amp

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Setup
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()
print(f"Device: {device}")

# ============================================================
# Configuration
# ============================================================
@dataclass
class Config:
    # Dataset paths
    WDP_PATH: str = "/content/webdesignprototypicality"
    WAD20_PATH: str = "/content/WAD20"
    RG_PATH: str = "/content/ReineckeGajos"

    WAD20_RATING_COL: str = "aesthetics"

    # Model config (optimized)
    HF_MODEL: str = "openai/clip-vit-base-patch16"
    IMG_SIZE: int = 224
    BASE_LR: float = 3e-6
    HEAD_LR: float = 2e-4
    WEIGHT_DECAY: float = 0.02
    BATCH_SIZE: int = 16
    EPOCHS_HEAD: int = 5
    EPOCHS_PARTIAL: int = 5
    EPOCHS_FULL: int = 5
    WARMUP_RATIO: float = 0.1

    VAL_SIZE: float = 0.15

    USE_TTA: bool = False
    BOOTSTRAP_N: int = 2000

    RESULTS_DIR: str = "/content/drive/MyDrive/results/vit_benchmark/cross_dataset"
    CACHE_DIR: str = "/content/cache_crossdataset"

    SEED: int = 42

    def __post_init__(self):
        from google.colab import drive
        drive.mount('/content/drive')

        if not os.path.exists(self.WDP_PATH):
            os.system(f'unzip -q /content/drive/MyDrive/datasets/webdesignprototypicality.zip -d {self.WDP_PATH}')
        if not os.path.exists(self.WAD20_PATH):
            os.makedirs(self.WAD20_PATH, exist_ok=True)
            os.system(f'unzip -q /content/drive/MyDrive/datasets/WAD20.zip -d {self.WAD20_PATH}')
        if not os.path.exists(self.RG_PATH):
            os.makedirs(self.RG_PATH, exist_ok=True)
            os.system(f'unzip -q /content/drive/MyDrive/datasets/ReineckeGajos.zip -d {self.RG_PATH}')

        for d in [self.RESULTS_DIR, self.CACHE_DIR]:
            os.makedirs(d, exist_ok=True)

        self.exp_name = f"cross_transfer_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.exp_dir = os.path.join(self.RESULTS_DIR, self.exp_name)
        os.makedirs(self.exp_dir, exist_ok=True)

CFG = Config()

# ============================================================
# Logging
# ============================================================
class Logger:
    def __init__(self, log_dir: str, name: str):
        self.log_file = os.path.join(log_dir, f"{name}.log")

    def log(self, msg: str):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] {msg}")
        with open(self.log_file, "a") as f:
            f.write(f"[{ts}] {msg}\n")

logger = Logger(CFG.exp_dir, CFG.exp_name)

# ============================================================
# Loss Function
# ============================================================
def loss_function(pred, target):
    return F.mse_loss(pred, target)

# ============================================================
# Dataset Loaders
# ============================================================

def load_wdp_data():
    logger.log("Loading WDP...")
    categories = ["banks", "fashion", "homeware", "universities"]
    rows = []

    for cat in categories:
        ratings_file = os.path.join(CFG.WDP_PATH, cat, f"ratings.avg.{cat}.txt")
        screenshot_dir = os.path.join(CFG.WDP_PATH, cat, "screenshots")

        if not os.path.exists(ratings_file):
            continue

        df = pd.read_csv(ratings_file, sep="\t")
        df["category"] = cat
        df["dataset"] = "WDP"
        df["screenshot_path"] = df["stimulusId"].apply(lambda s: os.path.join(screenshot_dir, s))
        df["rating"] = df["AE"]
        df = df[df["screenshot_path"].apply(os.path.exists)]
        rows.append(df[["stimulusId", "category", "dataset", "screenshot_path", "rating"]])

    df_all = pd.concat(rows, ignore_index=True).drop_duplicates("screenshot_path").reset_index(drop=True)
    logger.log(f"  WDP: {len(df_all)} samples")
    return df_all


def load_wad20_data():
    logger.log("Loading WAD20...")

    survey_file = os.path.join(CFG.WAD20_PATH, "surveyData.csv")
    if not os.path.exists(survey_file):
        survey_file = os.path.join(CFG.WAD20_PATH, "WAD20", "surveyData.csv")

    image_dir = os.path.join(CFG.WAD20_PATH, "data")
    if not os.path.exists(image_dir):
        image_dir = os.path.join(CFG.WAD20_PATH, "WAD20", "data")

    df = pd.read_csv(survey_file)
    df_agg = df.groupby("website").agg({CFG.WAD20_RATING_COL: "mean"}).reset_index()
    df_agg.columns = ["website", "rating"]

    df_agg["stimulusId"] = df_agg["website"]
    df_agg["category"] = "mixed"
    df_agg["dataset"] = "WAD20"
    df_agg["screenshot_path"] = df_agg["website"].apply(lambda s: os.path.join(image_dir, s))
    df_agg = df_agg[df_agg["screenshot_path"].apply(os.path.exists)]

    logger.log(f"  WAD20: {len(df_agg)} samples")
    return df_agg[["stimulusId", "category", "dataset", "screenshot_path", "rating"]]


def load_rg_data():
    logger.log("Loading R&G...")

    means_files = [
        os.path.join(CFG.RG_PATH, "preprocess", "train_means_list.csv"),
        os.path.join(CFG.RG_PATH, "ReineckeGajos", "preprocess", "train_means_list.csv"),
    ]
    means_file = next((f for f in means_files if os.path.exists(f)), None)

    df = pd.read_csv(means_file)

    def resolve_path(img_path):
        basename = os.path.basename(img_path.lstrip("/"))
        lang = "english" if "english" in img_path.lower() else "foreign"
        candidates = [
            os.path.join(CFG.RG_PATH, "images", lang, basename),
            os.path.join(CFG.RG_PATH, "ReineckeGajos", "images", lang, basename),
        ]
        return next((c for c in candidates if os.path.exists(c)), None)

    df["screenshot_path"] = df["image"].apply(resolve_path)
    df = df[df["screenshot_path"].notna()].reset_index(drop=True)

    df["stimulusId"] = df["image"].apply(os.path.basename)
    df["category"] = df["image"].apply(lambda x: "english" if "english" in x.lower() else "foreign")
    df["dataset"] = "R&G"
    df["rating"] = df["mean_score"]

    logger.log(f"  R&G: {len(df)} samples")
    return df[["stimulusId", "category", "dataset", "screenshot_path", "rating"]]


def normalize_dataset(df):
    """Z-score ratings within one dataset; Pearson r is invariant to this."""
    df = df.copy()
    mu, sd = df["rating"].mean(), df["rating"].std()
    df["rating_norm"] = (df["rating"] - mu) / (sd + 1e-8)
    return df

# ============================================================
# Dataset Class
# ============================================================

def letterbox_square(img, size):
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (128, 128, 128))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.BICUBIC)


def get_cache_path(src_path, dataset):
    basename = os.path.splitext(os.path.basename(src_path))[0]
    return os.path.join(CFG.CACHE_DIR, f"{dataset}_{basename}_{CFG.IMG_SIZE}.jpg")


class AestheticsDataset(Dataset):
    def __init__(self, df, mean, std):
        self.df = df.reset_index(drop=True)
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = get_cache_path(row["screenshot_path"], row["dataset"])

        if os.path.exists(cache_path):
            img = Image.open(cache_path).convert("RGB")
        else:
            img = Image.open(row["screenshot_path"]).convert("RGB")
            img = letterbox_square(img, CFG.IMG_SIZE)
            img.save(cache_path, "JPEG", quality=95)
            img = Image.open(cache_path).convert("RGB")

        x = TF.to_tensor(img)
        x = (x - torch.tensor(self.mean).view(-1,1,1)) / torch.tensor(self.std).view(-1,1,1)
        y = torch.tensor([float(row["rating_norm"])], dtype=torch.float32)

        return {"pixel_values": x, "labels": y}


def precache_dataset(df, name):
    logger.log(f"  Caching {name}...")
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        cache_path = get_cache_path(row["screenshot_path"], row["dataset"])
        if not os.path.exists(cache_path):
            try:
                img = Image.open(row["screenshot_path"]).convert("RGB")
                img = letterbox_square(img, CFG.IMG_SIZE)
                img.save(cache_path, "JPEG", quality=95)
            except:
                pass

# ============================================================
# Metrics
# ============================================================

def compute_metrics(y_true, y_pred):
    return {
        "r": float(pearsonr(y_pred, y_true)[0]),
        "rho": float(spearmanr(y_pred, y_true)[0]),
        "rmse": float(np.sqrt(np.mean((y_pred - y_true)**2))),
    }


def bootstrap_ci(y_true, y_pred, n=2000, alpha=0.05):
    rng = np.random.default_rng(42)
    vals = [pearsonr(y_pred[idx := rng.integers(0, len(y_true), len(y_true))], y_true[idx])[0] for _ in range(n)]
    return np.percentile(vals, [100*alpha/2, 100*(1-alpha/2)])

# ============================================================
# Model
# ============================================================

def load_fresh_model():
    model = AutoModelForImageClassification.from_pretrained(
        CFG.HF_MODEL, num_labels=1, problem_type="regression", ignore_mismatched_sizes=True
    )
    model.to(device)

    try:
        proc = AutoImageProcessor.from_pretrained(CFG.HF_MODEL)
        mean, std = proc.image_mean, proc.image_std
    except:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    return model, mean, std


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    with amp.autocast('cuda', enabled=use_cuda):
        for batch in loader:
            x = batch["pixel_values"].to(device)
            preds.append(model(x).logits.squeeze(1).cpu().numpy())
    return np.concatenate(preds)

# ============================================================
# Training (3-Phase)
# ============================================================

def train_3phase(model, train_loader, val_loader):

    def freeze_all(m, freeze=True):
        for p in m.parameters():
            p.requires_grad = not freeze

    def get_head_params(m):
        return [n for n, _ in m.named_parameters()
                if any(n.startswith(p) for p in ["classifier", "head", "fc", "score"])]

    def unfreeze_head(m):
        freeze_all(m, True)
        for n, p in m.named_parameters():
            if n in get_head_params(m):
                p.requires_grad = True

    def unfreeze_last_k(m, k=4):
        freeze_all(m, True)
        for n, p in m.named_parameters():
            if n in get_head_params(m):
                p.requires_grad = True
        if hasattr(m, "vision_model") and hasattr(m.vision_model, "encoder"):
            blocks = list(m.vision_model.encoder.layers)
            for i in range(max(0, len(blocks)-k), len(blocks)):
                for p in blocks[i].parameters():
                    p.requires_grad = True

    def unfreeze_all(m):
        freeze_all(m, False)

    best_r, best_state = -float("inf"), None

    def run_phase(epochs, lr, unfreeze_fn):
        nonlocal best_r, best_state

        unfreeze_fn(model)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=CFG.WEIGHT_DECAY)
        steps = epochs * len(train_loader)
        sch = get_cosine_schedule_with_warmup(opt, int(CFG.WARMUP_RATIO * steps), steps)
        scaler = amp.GradScaler(enabled=use_cuda)
        patience = 0

        for epoch in range(1, epochs + 1):
            model.train()
            for batch in train_loader:
                x = batch["pixel_values"].to(device)
                y = batch["labels"].to(device)

                with amp.autocast('cuda', enabled=use_cuda):
                    loss = loss_function(model(x).logits, y)

                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
                sch.step()

            y_pred = predict(model, val_loader)
            y_true = val_loader.dataset.df["rating_norm"].values
            r = pearsonr(y_pred, y_true)[0]

            if r > best_r:
                best_r = r
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 5:
                    break

        if best_state:
            model.load_state_dict(best_state)

    run_phase(CFG.EPOCHS_HEAD, CFG.HEAD_LR, unfreeze_head)
    run_phase(CFG.EPOCHS_PARTIAL, CFG.BASE_LR, lambda m: unfreeze_last_k(m, 4))
    run_phase(CFG.EPOCHS_FULL, CFG.BASE_LR, unfreeze_all)

    return model, best_r

# ============================================================
# Main
# ============================================================

def main():
    logger.log("="*70)
    logger.log("CROSS-DATASET TRANSFER (Off-Diagonal Only)")
    logger.log("Train on A → Test on B (where A ≠ B)")
    logger.log("="*70)

    # Load all datasets
    datasets = {
        "WDP": normalize_dataset(load_wdp_data()),
        "WAD20": normalize_dataset(load_wad20_data()),
        "R&G": normalize_dataset(load_rg_data()),
    }

    # Precache
    logger.log("\nPrecaching images...")
    for name, df in datasets.items():
        precache_dataset(df, name)

    # Transfer matrix (off-diagonal only)
    results = {}

    for train_name, df_train_full in datasets.items():
        logger.log(f"\n{'='*60}")
        logger.log(f"TRAINING ON: {train_name} ({len(df_train_full)} samples)")
        logger.log(f"{'='*60}")

        # Split train/val
        stratify = df_train_full["category"] if df_train_full["category"].nunique() > 1 else None
        df_train, df_val = train_test_split(
            df_train_full, test_size=CFG.VAL_SIZE, random_state=CFG.SEED, stratify=stratify
        )

        logger.log(f"  Train: {len(df_train)}, Val: {len(df_val)}")

        # Fresh model
        set_seed(CFG.SEED)
        model, mean, std = load_fresh_model()

        # Train
        train_ds = AestheticsDataset(df_train, mean, std)
        val_ds = AestheticsDataset(df_val, mean, std)

        train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=0, pin_memory=True)

        model, best_val_r = train_3phase(model, train_loader, val_loader)
        logger.log(f"  Best val r: {best_val_r:.4f}")

        # Test on OTHER datasets only (skip diagonal)
        results[train_name] = {}

        for test_name, df_test in datasets.items():
            if test_name == train_name:
                continue  # Skip diagonal

            test_ds = AestheticsDataset(df_test, mean, std)
            test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                                    num_workers=0, pin_memory=True)

            y_pred = predict(model, test_loader)
            y_true = df_test["rating_norm"].values

            metrics = compute_metrics(y_true, y_pred)
            r_ci = bootstrap_ci(y_true, y_pred)

            results[train_name][test_name] = {
                "r": metrics["r"],
                "rho": metrics["rho"],
                "rmse": metrics["rmse"],
                "r_ci": r_ci.tolist(),
                "n_train": len(df_train_full),
                "n_test": len(df_test),
            }

            logger.log(f"  → {test_name}: r={metrics['r']:.3f} [{r_ci[0]:.3f}, {r_ci[1]:.3f}] (n={len(df_test)})")

        del model
        torch.cuda.empty_cache()

    # Save
    results_path = os.path.join(CFG.exp_dir, "results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    logger.log(f"\nSaved: {results_path}")

    # Summary
    logger.log("\n" + "="*70)
    logger.log("CROSS-DATASET TRANSFER MATRIX")
    logger.log("="*70)
    logger.log(f"\n{'Train → Test':<20} {'n_train':>8} {'n_test':>8} {'r':>8} {'95% CI':>16}")
    logger.log("-"*65)

    for train_name in ["WDP", "WAD20", "R&G"]:
        for test_name in ["WDP", "WAD20", "R&G"]:
            if test_name == train_name:
                continue
            res = results[train_name][test_name]
            ci = res["r_ci"]
            logger.log(f"{train_name} → {test_name:<8} {res['n_train']:>8} {res['n_test']:>8} {res['r']:>8.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")

    logger.log("\n" + "="*70)
    logger.log("DONE")
    logger.log("="*70)


if __name__ == "__main__":
    main()

Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[08:21:54] ======================================================================
[08:21:55] CROSS-DATASET TRANSFER (Off-Diagonal Only)
[08:21:55] Train on A → Test on B (where A ≠ B)
[08:21:55] ======================================================================
[08:21:55] Loading WDP...
[08:21:55]   WDP: 3136 samples
[08:21:55] Loading WAD20...
[08:21:55]   WAD20: 540 samples
[08:21:55] Loading R&G...
[08:21:55]   R&G: 300 samples
[08:21:55] 
Precaching images...
[08:21:55]   Caching WDP...


  0%|          | 0/3136 [00:00<?, ?it/s]

[08:35:53]   Caching WAD20...


  0%|          | 0/540 [00:00<?, ?it/s]

[08:36:44]   Caching R&G...


...

[08:42:18]   Best val r: 0.6635
[08:42:21]   → WAD20: r=0.599 [0.544, 0.648] (n=540)
[08:42:23]   → R&G: r=0.623 [0.539, 0.702] (n=300)
[08:42:23] 
[08:42:23] TRAINING ON: WAD20 (540 samples)
[08:42:23] ============================================================
[08:42:23]   Train: 459, Val: 81


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[08:43:18]   Best val r: 0.6121
[08:43:31]   → WDP: r=0.109 [0.073, 0.146] (n=3136)
[08:43:33]   → R&G: r=0.641 [0.553, 0.719] (n=300)
[08:43:33] 
[08:43:33] TRAINING ON: R&G (300 samples)
[08:43:33] ============================================================
[08:43:33]   Train: 255, Val: 45


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[08:44:04]   Best val r: 0.6526
[08:44:17]   → WDP: r=0.271 [0.233, 0.312] (n=3136)
[08:44:20]   → WAD20: r=0.543 [0.486, 0.599] (n=540)
[08:44:20] 
Saved: /content/drive/MyDrive/results/vit_benchmark/cross_dataset/cross_transfer_20260128_082154/results.json
[08:44:20] 
[08:44:20] CROSS-DATASET TRANSFER MATRIX
[08:44:20] ======================================================================
[08:44:20] 
Train → Test          n_train   n_test        r           95% CI
[08:44:20] -----------------------------------------------------------------
[08:44:20] WDP → WAD20        3136      540    0.599 [0.544, 0.648]
[08:44:20] WDP → R&G          3136      300    0.623 [0.539, 0.702]
[08:44:20] WAD20 → WDP           540     3136    0.109 [0.073, 0.146]
[08:44:20] WAD20 → R&G           540      300    0.641 [0.553, 0.719]
[08:44:20] R&G → WDP           300     3136    0.271 [0.233, 0.312]
[08:44:20] R&G → WAD20         300      540    0.543 [0.486, 0.599]
[08:44:20] 
[08:44:20] DONE
[08:44:20] =